# sebal-et-gee — Day 3: Albedo, Soil Heat Flux G, and Hot/Cold Anchor Pixels

**Today's deliverables:**
1. True per-pixel broadband albedo from Landsat (Tasumi 2008)
2. Updated Net Radiation Rn using real albedo
3. Soil heat flux G (Bastiaanssen 1998)
4. Hot and cold anchor pixels — the heart of SEBAL

**SEBAL energy balance progress:**

$$R_n - G - H = \lambda ET$$

After today: Rn ✓, G ✓, H anchor-pixel boundary conditions ✓. Day 4 will solve H iteratively and derive daily ET.

## 1. Setup (rebuild mosaic from Day 2)

In [2]:
!pip install -q geemap geopandas

import ee
import geemap
import geopandas as gpd
import pandas as pd

EE_PROJECT = 'et-thrace'  # <-- REPLACE

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print('Earth Engine ready.')


Earth Engine ready.


In [3]:
from google.colab import files
print('Upload thrace_boundary.geojson:')
uploaded = files.upload()

aoi_path = [f for f in uploaded if f.endswith(('.geojson', '.json'))][0]
aoi_gdf = gpd.read_file(aoi_path)
aoi_ee = geemap.geopandas_to_ee(aoi_gdf)
aoi_geom = aoi_ee.geometry()

stations = pd.DataFrame([
    {'name': 'Edirne',      'lat': 41.6667, 'lon': 26.5667, 'elevation_m':  51},
    {'name': 'Kirklareli',  'lat': 41.7333, 'lon': 27.2167, 'elevation_m': 232},
    {'name': 'Luleburgaz',  'lat': 41.4000, 'lon': 27.3500, 'elevation_m':  46},
    {'name': 'Uzunkopru',   'lat': 41.2667, 'lon': 26.6833, 'elevation_m':  22},
    {'name': 'Ipsala',      'lat': 40.9167, 'lon': 26.3833, 'elevation_m':  10},
    {'name': 'Corlu',       'lat': 41.1500, 'lon': 27.8000, 'elevation_m': 183},
    {'name': 'Tekirdag',    'lat': 40.9833, 'lon': 27.5167, 'elevation_m':   3},
    {'name': 'Sariyer',     'lat': 41.1667, 'lon': 29.0500, 'elevation_m':  59},
])
stations_ee = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([r['lon'], r['lat']]),
               {'name': r['name'], 'elevation_m': int(r['elevation_m'])})
    for _, r in stations.iterrows()
])
print('AOI + stations loaded.')


Upload thrace_boundary.geojson:


Saving thrace_stations.csv to thrace_stations.csv
Saving thrace_boundary.geojson to thrace_boundary.geojson
AOI + stations loaded.


In [4]:
# Rebuild the Day 2 Landsat mosaic
def apply_scale_factors(image):
    optical = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermal_k = image.select('ST_B10').multiply(0.00341802).add(149.0).rename('LST_K')
    thermal_c = thermal_k.subtract(273.15).rename('LST_C')
    return image.addBands(optical, None, True).addBands(thermal_k).addBands(thermal_c)

def mask_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud = qa.bitwiseAnd(1 << 3).neq(0)
    shadow = qa.bitwiseAnd(1 << 4).neq(0)
    return image.updateMask(cloud.Or(shadow).Not())

COMMON_BANDS = ['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7','ST_B10','QA_PIXEL']
ANCHOR = '2023-07-20'
WINDOW_DAYS = 14

anchor = ee.Date(ANCHOR)
start = anchor.advance(-WINDOW_DAYS // 2, 'day')
end = anchor.advance(WINDOW_DAYS // 2, 'day')

l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterBounds(aoi_geom)
      .filterDate(start, end).filter(ee.Filter.lt('CLOUD_COVER', 80)).select(COMMON_BANDS))
l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterBounds(aoi_geom)
      .filterDate(start, end).filter(ee.Filter.lt('CLOUD_COVER', 80)).select(COMMON_BANDS))

merged = l8.merge(l9).map(mask_clouds).map(apply_scale_factors).sort('CLOUD_COVER')
mosaic = merged.mosaic().clip(aoi_geom)
print(f'Mosaic rebuilt. Scenes: {merged.size().getInfo()}')


Mosaic rebuilt. Scenes: 14


## 2. Tasumi (2008) broadband albedo

Narrowband-to-broadband conversion from at-surface reflectance for Landsat 8/9 OLI:

$$\alpha = 0.300 \cdot \rho_{B2} + 0.277 \cdot \rho_{B3} + 0.233 \cdot \rho_{B4} + 0.143 \cdot \rho_{B5} + 0.036 \cdot \rho_{B6} + 0.012 \cdot \rho_{B7}$$

Coefficients sum to 1.001, approximating incident solar energy distribution across shortwave bands. This is the standard METRIC/SEBAL formulation for Landsat 8 (Tasumi et al., 2008, *J. Hydrol. Eng.* 13:51–63).

In [5]:
def compute_albedo(img):
    """Tasumi 2008 broadband albedo for Landsat 8/9 OLI."""
    coeffs = {
        'SR_B2': 0.300, 'SR_B3': 0.277, 'SR_B4': 0.233,
        'SR_B5': 0.143, 'SR_B6': 0.036, 'SR_B7': 0.012,
    }
    terms = [img.select(b).multiply(c) for b, c in coeffs.items()]
    albedo = terms[0]
    for t in terms[1:]:
        albedo = albedo.add(t)
    return albedo.rename('albedo').clamp(0.0, 1.0)

albedo = compute_albedo(mosaic)

# Station-level sanity check
stn_alb = albedo.reduceRegions(collection=stations_ee, reducer=ee.Reducer.mean(), scale=30)
df_alb = geemap.ee_to_df(stn_alb)[['name', 'mean']].rename(columns={'mean': 'albedo'})
df_alb['albedo'] = df_alb['albedo'].round(3)
print('Station-level broadband albedo:')
df_alb


Station-level broadband albedo:


,name,albedo
0,Edirne,0.159
1,Kirklareli,0.136
2,Luleburgaz,0.129
3,Uzunkopru,0.145
4,Ipsala,0.090
5,Corlu,0.126
6,Tekirdag,0.116
7,Sariyer,0.086


**Expected albedo ranges for Thrace summer:**
- Dense vegetation / irrigated fields: 0.15–0.22
- Dry cropland / bare soil: 0.25–0.35
- Urban: 0.15–0.25
- Water: 0.03–0.08

If any station returns albedo < 0.05 or > 0.40, investigate before continuing.

## 3. Recompute Net Radiation Rn with real albedo + ERA5-Land

In [6]:
era5_vars = ['temperature_2m', 'dewpoint_temperature_2m',
             'u_component_of_wind_10m', 'v_component_of_wind_10m',
             'surface_solar_radiation_downwards_hourly',
             'surface_thermal_radiation_downwards_hourly',
             'surface_pressure']

n_days = int(end.difference(start, 'day').getInfo())
daily_era5 = []
for i in range(n_days):
    d = start.advance(i, 'day')
    h0 = d.update(hour=10, minute=0, second=0)
    daily_era5.append(ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY')
                      .filterDate(h0, h0.advance(1, 'hour'))
                      .select(era5_vars).first())
era5_mean = ee.ImageCollection(daily_era5).mean().clip(aoi_geom)

T_air_K = era5_mean.select('temperature_2m').rename('T_air_K')
T_air_C = T_air_K.subtract(273.15).rename('T_air_C')
u10 = era5_mean.select('u_component_of_wind_10m')
v10 = era5_mean.select('v_component_of_wind_10m')
wind_10m = u10.pow(2).add(v10.pow(2)).sqrt().rename('wind_10m')
Rs_in = era5_mean.select('surface_solar_radiation_downwards_hourly').divide(3600).rename('Rs_in')
Rl_in = era5_mean.select('surface_thermal_radiation_downwards_hourly').divide(3600).rename('Rl_in')
P_surf = era5_mean.select('surface_pressure').rename('P_surf')

# Outgoing longwave from mosaic LST + NDVI-based emissivity
SIGMA = 5.670374419e-8
ndvi = mosaic.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
emissivity = ndvi.multiply(0.01).add(0.985).clamp(0.96, 0.99).rename('emissivity')
LST_K = mosaic.select('LST_K')
Rl_out = emissivity.multiply(SIGMA).multiply(LST_K.pow(4)).rename('Rl_out')

# Real Rn
Rn = (Rs_in.multiply(ee.Image(1).subtract(albedo))
      .add(Rl_in)
      .subtract(Rl_out)
      .rename('Rn'))

stn_rn = Rn.reduceRegions(collection=stations_ee, reducer=ee.Reducer.mean(), scale=30)
df_rn = geemap.ee_to_df(stn_rn)[['name', 'mean']].rename(columns={'mean': 'Rn_Wm2'})
df_rn['Rn_Wm2'] = df_rn['Rn_Wm2'].round(1)
print('Station-level Rn (with true albedo):')
df_rn


Station-level Rn (with true albedo):


,name,Rn_Wm2
0,Edirne,550.6
1,Kirklareli,605.8
2,Luleburgaz,602.9
3,Uzunkopru,589.4
4,Ipsala,682.2
5,Corlu,596.9
6,Tekirdag,615.1
7,Sariyer,687.4


**Compare to Day 2 Rn values.** They should be similar but slightly different (5–15%) per station depending on how each pixel's true albedo differs from the 0.23 placeholder.

## 4. Soil Heat Flux G (Bastiaanssen 1998)

$$\frac{G}{R_n} = \frac{T_s - 273.15}{\alpha} \cdot (0.0038 \alpha + 0.0074 \alpha^2)(1 - 0.98 \cdot \text{NDVI}^4)$$

where $T_s$ is land surface temperature in K. This is the classic Bastiaanssen (1998, *J. Hydrol.* 212–213:198–212) empirical form, widely used in SEBAL/METRIC. For water bodies (NDVI < 0), G/Rn is set to 0.5.

In [7]:
def compute_G(Rn, LST_K, albedo, ndvi):
    T_C = LST_K.subtract(273.15)
    term1 = T_C.divide(albedo)
    term2 = albedo.multiply(0.0038).add(albedo.pow(2).multiply(0.0074))
    term3 = ndvi.pow(4).multiply(0.98).multiply(-1).add(1)
    g_ratio = term1.multiply(term2).multiply(term3).rename('g_ratio')

    # Water mask: NDVI < 0 → G/Rn = 0.5
    water = ndvi.lt(0)
    g_ratio = g_ratio.where(water, 0.5)
    g_ratio = g_ratio.clamp(0.05, 0.5)  # physical bounds

    G = Rn.multiply(g_ratio).rename('G')
    return G, g_ratio

G, g_ratio = compute_G(Rn, LST_K, albedo, ndvi)

stn_g = G.reduceRegions(collection=stations_ee, reducer=ee.Reducer.mean(), scale=30)
df_g = geemap.ee_to_df(stn_g)[['name', 'mean']].rename(columns={'mean': 'G_Wm2'})
df_g['G_Wm2'] = df_g['G_Wm2'].round(1)
print('Station-level soil heat flux G (W/m²):')
df_g


Station-level soil heat flux G (W/m²):


,name,G_Wm2
0,Edirne,133.9
1,Kirklareli,118.4
2,Luleburgaz,128.6
3,Uzunkopru,132.7
4,Ipsala,85.3
5,Corlu,130.7
6,Tekirdag,127.2
7,Sariyer,78.2


**Expected G range:** typically 50–200 W/m² for vegetated/cropland surfaces. Higher for bare soil, lower for dense vegetation. G/Rn ratio should be 0.1–0.4 at stations.

## 5. Available Energy (Rn − G)

This is the total energy available at the surface for partitioning between sensible heat H and latent heat λET. Day 4 will determine how that split happens.

In [8]:
Rn_minus_G = Rn.subtract(G).rename('Rn_minus_G')

stn_rng = Rn_minus_G.reduceRegions(collection=stations_ee, reducer=ee.Reducer.mean(), scale=30)
df_rng = geemap.ee_to_df(stn_rng)[['name', 'mean']].rename(columns={'mean': 'Rn-G_Wm2'})
df_rng['Rn-G_Wm2'] = df_rng['Rn-G_Wm2'].round(1)
print('Station-level available energy (Rn − G):')
df_rng


Station-level available energy (Rn − G):


,name,Rn-G_Wm2
0,Edirne,416.7
1,Kirklareli,487.3
2,Luleburgaz,474.3
3,Uzunkopru,456.7
4,Ipsala,596.9
5,Corlu,466.1
6,Tekirdag,487.9
7,Sariyer,609.2


## 6. Hot and Cold Anchor Pixel Selection — SEBAL's Core

SEBAL's inversion relies on two anchor pixels that bracket the physical range of ET:

- **Cold pixel**: well-watered agricultural field → assumed $H \approx 0$, so $\lambda ET = R_n - G$
- **Hot pixel**: dry, bare agricultural surface → assumed $\lambda ET \approx 0$, so $H = R_n - G$

These two pixels define the linear $dT$–$T_s$ relationship used to solve H across the image.

**Automatic selection strategy (Allen et al., 2013 "CIMEC" variant):**
- Mask to agricultural pixels only (NDVI > 0.15, not water, not urban)
- Cold candidates: high NDVI (top 5%), low LST (bottom 20% within those)
- Hot candidates: low NDVI (but still vegetated or bare, not water), high LST (top 5%)
- Pick the median pixel within each candidate pool to avoid outliers

In [9]:
# Agricultural mask: exclude water (NDVI<0), very low veg (<0.05), and extreme albedo (urban/built)
ag_mask = (ndvi.gt(0.05)
           .And(albedo.lt(0.30))
           .And(albedo.gt(0.10)))

ag_stack = ee.Image.cat([ndvi, LST_K.rename('LST_K'), albedo, Rn, G]).updateMask(ag_mask)

# Compute AOI-wide percentiles for NDVI and LST on agricultural pixels
pcts = ag_stack.reduceRegion(
    reducer=ee.Reducer.percentile([5, 20, 80, 95]),
    geometry=aoi_geom,
    scale=90,  # coarser for efficiency
    maxPixels=1e9,
    bestEffort=True,
)
pcts_info = pcts.getInfo()
print('Agricultural-pixel percentiles:')
for k, v in pcts_info.items():
    print(f'  {k}: {v:.3f}')


Agricultural-pixel percentiles:
  G_p20: 113.252
  G_p5: 81.251
  G_p80: 130.750
  G_p95: 134.242
  LST_K_p20: 313.755
  LST_K_p5: 308.758
  LST_K_p80: 320.746
  LST_K_p95: 323.740
  NDVI_p20: 0.260
  NDVI_p5: 0.221
  NDVI_p80: 0.525
  NDVI_p95: 0.736
  Rn_p20: 559.008
  Rn_p5: 531.016
  Rn_p80: 636.994
  Rn_p95: 678.994
  albedo_p20: 0.112
  albedo_p5: 0.103
  albedo_p80: 0.160
  albedo_p95: 0.182


In [11]:
# --- Cell 7: Percentiles on agricultural pixels ---
ag_mask = (ndvi.gt(0.05)
           .And(albedo.lt(0.30))
           .And(albedo.gt(0.10)))

ag_stack = ee.Image.cat([ndvi, LST_K.rename('LST_K'), albedo, Rn, G]).updateMask(ag_mask)

pcts = ag_stack.reduceRegion(
    reducer=ee.Reducer.percentile([5, 20, 80, 95]),
    geometry=aoi_geom,
    scale=90,
    maxPixels=1e9,
    bestEffort=True,
)
pcts_info = pcts.getInfo()

# DEBUG: see exact key names
print('Available percentile keys:')
for k in sorted(pcts_info.keys()):
    print(f'  {k}: {pcts_info[k]:.3f}' if pcts_info[k] is not None else f'  {k}: None')

Available percentile keys:
  G_p20: 113.252
  G_p5: 81.251
  G_p80: 130.750
  G_p95: 134.242
  LST_K_p20: 313.755
  LST_K_p5: 308.758
  LST_K_p80: 320.746
  LST_K_p95: 323.740
  NDVI_p20: 0.260
  NDVI_p5: 0.221
  NDVI_p80: 0.525
  NDVI_p95: 0.736
  Rn_p20: 559.008
  Rn_p5: 531.016
  Rn_p80: 636.994
  Rn_p95: 678.994
  albedo_p20: 0.112
  albedo_p5: 0.103
  albedo_p80: 0.160
  albedo_p95: 0.182


In [12]:
# --- Cell 8: Cold and hot anchor pixel selection (fixed band names) ---

# Cold candidates: NDVI > p95 AND LST < p20
cold_mask = (ag_stack.select('NDVI').gt(pcts_info['NDVI_p95'])
             .And(ag_stack.select('LST_K').lt(pcts_info['LST_K_p20'])))

# Hot candidates: NDVI between p5 and p20 AND LST > p95
hot_mask = (ag_stack.select('NDVI').gt(pcts_info['NDVI_p5'])
            .And(ag_stack.select('NDVI').lt(pcts_info['NDVI_p20']))
            .And(ag_stack.select('LST_K').gt(pcts_info['LST_K_p95'])))

cold_pixels = ag_stack.updateMask(cold_mask)
hot_pixels = ag_stack.updateMask(hot_mask)

cold_stats = cold_pixels.reduceRegion(
    reducer=ee.Reducer.median(), geometry=aoi_geom, scale=90,
    maxPixels=1e9, bestEffort=True
).getInfo()
hot_stats = hot_pixels.reduceRegion(
    reducer=ee.Reducer.median(), geometry=aoi_geom, scale=90,
    maxPixels=1e9, bestEffort=True
).getInfo()

def fmt(d):
    return {k: round(v, 3) if v is not None else None for k, v in d.items()}

print('COLD anchor (well-watered, high NDVI, low LST):')
print(' ', fmt(cold_stats))
print()
print('HOT anchor (dry bare, low NDVI, high LST):')
print(' ', fmt(hot_stats))

COLD anchor (well-watered, high NDVI, low LST):
  {'G': 49.376, 'LST_K': 304.813, 'NDVI': 0.852, 'Rn': 694.505, 'albedo': 0.104}

HOT anchor (dry bare, low NDVI, high LST):
  {'G': 135.219, 'LST_K': 324.687, 'NDVI': 0.241, 'Rn': 523.495, 'albedo': 0.163}


**Physical sanity checks:**
- Cold LST should be 290–305 K (cool surface, ET active)
- Hot LST should be 315–330 K (~42–57 °C surface, ET shut down)
- LST gap (hot − cold): typically 10–25 K for Mediterranean summer
- Cold albedo: 0.15–0.22 (green canopy)
- Hot albedo: 0.20–0.30 (dry bare)

These two anchors are the boundary conditions that Day 4's H iteration will linearize between.

## 7. Visualize everything on the map

In [13]:
m = geemap.Map(center=[41.3, 27.5], zoom=8)
m.add_basemap('SATELLITE')

m.addLayer(mosaic, {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0, 'max': 0.3}, 'Landsat RGB')
m.addLayer(ndvi, {'min': 0, 'max': 0.9, 'palette': ['brown', 'white', 'green']}, 'NDVI', False)
m.addLayer(albedo, {'min': 0.05, 'max': 0.35,
                    'palette': ['000080', '00ffff', 'ffff00', 'ff0000']},
           'Albedo (Tasumi 2008)', False)
m.addLayer(Rn, {'min': 300, 'max': 800,
                'palette': ['000080', '0000ff', '00ffff', 'ffff00', 'ff8000', 'ff0000']},
           'Rn (W/m²)')
m.addLayer(G, {'min': 20, 'max': 250,
               'palette': ['blue', 'cyan', 'yellow', 'red']},
           'G (W/m²)', False)
m.addLayer(Rn_minus_G, {'min': 200, 'max': 700,
                        'palette': ['purple', 'blue', 'green', 'yellow', 'red']},
           'Rn − G (available energy)', False)

# Highlight cold/hot candidate pixels
m.addLayer(cold_pixels.select('LST_K').mask().selfMask(),
           {'palette': ['00ffff']}, 'Cold candidate pixels', False)
m.addLayer(hot_pixels.select('LST_K').mask().selfMask(),
           {'palette': ['ff00ff']}, 'Hot candidate pixels', False)

m.addLayer(aoi_ee, {'color': 'yellow'}, 'Thrace AOI')
m.addLayer(stations_ee, {'color': 'red'}, '8 stations')

m.addLayerControl()
m


Map(center=[41.3, 27.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

## 8. Day 3 deliverables checklist

- [ ] Tasumi 2008 albedo computed; station values in 0.10–0.30 range
- [ ] Rn recomputed with true albedo; station values 450–750 W/m²
- [ ] Soil heat flux G computed; station values 50–200 W/m²
- [ ] Hot and cold anchor pixel statistics physically plausible
- [ ] Cold LST < Hot LST by 10–25 K

**Save for Day 4:** keep these variable names in memory — Day 4 will use `Rn`, `G`, `Rn_minus_G`, `LST_K`, `T_air_K`, `wind_10m`, `albedo`, `ndvi`, `cold_stats`, `hot_stats`.

**Day 4 preview:**
1. Aerodynamic resistance $r_{ah}$ using wind log profile
2. Solve H iteratively using the cold/hot anchor linear $dT$-$T_s$ relation
3. Latent heat flux $\lambda ET = R_n - G - H$
4. Instantaneous → daily ET via Evaporative Fraction (EF) method
5. **Final output: daily ET map over Thrace in mm/day** — the MVP deliverable.